In [1]:
# Phase 7: Feature Engineering & Target Construction

import pandas as pd
import numpy as np
from pathlib import Path

# Load the cleaned transaction dataset
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "online_retail_clean.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
print("Minimum transaction date:", df["InvoiceDate"].min())
print("Maximum transaction date:", df["InvoiceDate"].max())
print("Unique customers:", df["Customer ID"].nunique())

Dataset shape: (1033036, 33)
Minimum transaction date: 2009-12-01 07:45:00
Maximum transaction date: 2011-12-09 12:50:00
Unique customers: 5942


In [2]:
# Identify qualifying purchase transactions
purchase_df = df.loc[df["is_qualifying_purchase"]].copy()

# Ensure transaction dates are datetime values
purchase_df["InvoiceDate"] = pd.to_datetime(
    purchase_df["InvoiceDate"],
    errors="raise"
)

# Determine the earliest and latest qualifying purchase dates
first_purchase_date = purchase_df["InvoiceDate"].min()
last_purchase_date = purchase_df["InvoiceDate"].max()

# Calculate the first monthly snapshot with a complete 180-day history
earliest_possible_snapshot = first_purchase_date + pd.Timedelta(days=180)

first_snapshot = earliest_possible_snapshot.to_period("M").to_timestamp()

if first_snapshot < earliest_possible_snapshot.normalize():
    first_snapshot += pd.offsets.MonthBegin(1)

# Calculate the last monthly snapshot with a complete 90-day outcome window
latest_possible_snapshot = last_purchase_date - pd.Timedelta(days=90)
last_snapshot = latest_possible_snapshot.to_period("M").to_timestamp()

# Generate monthly snapshot dates
snapshot_dates = pd.date_range(
    start=first_snapshot,
    end=last_snapshot,
    freq="MS"
)

print("Qualifying purchase records:", len(purchase_df))
print("Unique qualifying-purchase customers:", purchase_df["Customer ID"].nunique())
print("First qualifying purchase:", first_purchase_date)
print("Last qualifying purchase:", last_purchase_date)
print("First snapshot date:", first_snapshot)
print("Last snapshot date:", last_snapshot)
print("Number of monthly snapshots:", len(snapshot_dates))
print("\nSnapshot dates:")
print(snapshot_dates)


Qualifying purchase records: 776504
Unique qualifying-purchase customers: 5852
First qualifying purchase: 2009-12-01 07:45:00
Last qualifying purchase: 2011-12-09 12:50:00
First snapshot date: 2010-06-01 00:00:00
Last snapshot date: 2011-09-01 00:00:00
Number of monthly snapshots: 16

Snapshot dates:
DatetimeIndex(['2010-06-01', '2010-07-01', '2010-08-01', '2010-09-01',
               '2010-10-01', '2010-11-01', '2010-12-01', '2011-01-01',
               '2011-02-01', '2011-03-01', '2011-04-01', '2011-05-01',
               '2011-06-01', '2011-07-01', '2011-08-01', '2011-09-01'],
              dtype='datetime64[us]', freq='MS')


In [3]:
# Build a preliminary temporal snapshot audit

snapshot_audit = []

for snapshot_date in snapshot_dates:

    feature_start = snapshot_date - pd.Timedelta(days=180)
    target_end = snapshot_date + pd.Timedelta(days=90)

    # Customers with qualifying purchases in the feature window
    feature_customers = set(
        purchase_df.loc[
            (purchase_df["InvoiceDate"] >= feature_start) &
            (purchase_df["InvoiceDate"] < snapshot_date),
            "Customer ID"
        ].unique()
    )

    # Customers with qualifying purchases in the target window
    target_customers = set(
        purchase_df.loc[
            (purchase_df["InvoiceDate"] >= snapshot_date) &
            (purchase_df["InvoiceDate"] < target_end),
            "Customer ID"
        ].unique()
    )

    # Positive outcomes among eligible customers
    positive_customers = feature_customers.intersection(target_customers)

    eligible_count = len(feature_customers)
    positive_count = len(positive_customers)
    negative_count = eligible_count - positive_count

    snapshot_audit.append({
        "snapshot_date": snapshot_date,
        "feature_start": feature_start,
        "target_end": target_end,
        "eligible_customers": eligible_count,
        "positive_customers": positive_count,
        "negative_customers": negative_count,
        "positive_rate": (
            positive_count / eligible_count
            if eligible_count > 0 else np.nan
        )
    })

snapshot_audit_df = pd.DataFrame(snapshot_audit)

display(
    snapshot_audit_df.style.format({
        "positive_rate": "{:.2%}"
    })
)

print("Total customer-snapshot observations:",
      snapshot_audit_df["eligible_customers"].sum())

print("Total positive observations:",
      snapshot_audit_df["positive_customers"].sum())

print("Total negative observations:",
      snapshot_audit_df["negative_customers"].sum())

,snapshot_date,feature_start,target_end,eligible_customers,positive_customers,negative_customers,positive_rate
0,2010-06-01 00:00:00,2009-12-03 00:00:00,2010-08-30 00:00:00,2652,1327,1325,50.04%
1,2010-07-01 00:00:00,2010-01-02 00:00:00,2010-09-29 00:00:00,2737,1394,1343,50.93%
2,2010-08-01 00:00:00,2010-02-02 00:00:00,2010-10-30 00:00:00,2813,1604,1209,57.02%
3,2010-09-01 00:00:00,2010-03-05 00:00:00,2010-11-30 00:00:00,2822,1750,1072,62.01%
4,2010-10-01 00:00:00,2010-04-04 00:00:00,2010-12-30 00:00:00,2899,1680,1219,57.95%
5,2010-11-01 00:00:00,2010-05-05 00:00:00,2011-01-30 00:00:00,3196,1577,1619,49.34%
6,2010-12-01 00:00:00,2010-06-04 00:00:00,2011-03-01 00:00:00,3465,1318,2147,38.04%
7,2011-01-01 00:00:00,2010-07-05 00:00:00,2011-04-01 00:00:00,3403,1284,2119,37.73%
8,2011-02-01 00:00:00,2010-08-05 00:00:00,2011-05-02 00:00:00,3348,1328,2020,39.67%
9,2011-03-01 00:00:00,2010-09-02 00:00:00,2011-05-30 00:00:00,3338,1414,1924,42.36%


Total customer-snapshot observations: 47933
Total positive observations: 23533
Total negative observations: 24400


In [4]:
# Construct historical customer features

# Recreate analytical invoice groups consistently with Phase 5/6
invoice_group_cols = [
    "Invoice",
    "InvoiceDate",
    "Customer ID",
    "Country"
]

purchase_df = purchase_df.copy()

purchase_df["invoice_group_id"] = (
    purchase_df.groupby(
        invoice_group_cols,
        sort=False,
        dropna=False
    ).ngroup()
)

snapshot_feature_frames = []

for snapshot_date in snapshot_dates:

    feature_start = snapshot_date - pd.Timedelta(days=180)

    # Restrict transactions to the historical feature window
    window_df = purchase_df.loc[
        (purchase_df["InvoiceDate"] >= feature_start) &
        (purchase_df["InvoiceDate"] < snapshot_date)
    ].copy()

    # Aggregate historical purchasing behavior by customer
    customer_features = (
        window_df.groupby("Customer ID")
        .agg(
            last_purchase_date=("InvoiceDate", "max"),
            first_purchase_date=("InvoiceDate", "min"),
            frequency=("invoice_group_id", "nunique"),
            monetary=("qualifying_purchase_revenue", "sum"),
            purchase_line_count=("Invoice", "size"),
            unique_products=("StockCode", "nunique")
        )
        .reset_index()
    )

    # Add snapshot-specific information
    customer_features["snapshot_date"] = snapshot_date
    customer_features["feature_start"] = feature_start

    customer_features["recency_days"] = (
        snapshot_date - customer_features["last_purchase_date"]
    ).dt.days

    customer_features["customer_history_days"] = (
        customer_features["last_purchase_date"] -
        customer_features["first_purchase_date"]
    ).dt.days

    customer_features["avg_invoice_value"] = (
        customer_features["monetary"] /
        customer_features["frequency"]
    )

    snapshot_feature_frames.append(customer_features)

# Combine all monthly customer snapshots
customer_features_df = pd.concat(
    snapshot_feature_frames,
    ignore_index=True
)

print("Feature dataset shape:", customer_features_df.shape)
print("Unique customers:", customer_features_df["Customer ID"].nunique())
print("Snapshot observations:", customer_features_df.shape[0])
print("\nMissing values:")
print(customer_features_df.isna().sum())

display(customer_features_df.head())

Feature dataset shape: (47933, 12)
Unique customers: 5212
Snapshot observations: 47933

Missing values:
Customer ID              0
last_purchase_date       0
first_purchase_date      0
frequency                0
monetary                 0
purchase_line_count      0
unique_products          0
snapshot_date            0
feature_start            0
recency_days             0
customer_history_days    0
avg_invoice_value        0
dtype: int64


,Customer ID,last_purchase_date,first_purchase_date,frequency,monetary,purchase_line_count,unique_products,snapshot_date,feature_start,recency_days,customer_history_days,avg_invoice_value
0,12346.0,2010-03-02 13:08:00,2009-12-18 10:55:00,2,28.05,6,6,2010-06-01,2009-12-03,90,74,14.0250
1,12349.0,2010-04-29 13:20:00,2010-04-29 13:20:00,1,1068.52,46,46,2010-06-01,2009-12-03,32,0,1068.5200
2,12355.0,2010-05-21 11:59:00,2010-05-21 11:59:00,1,488.21,22,22,2010-06-01,2009-12-03,10,0,488.2100
3,12358.0,2009-12-08 07:59:00,2009-12-08 07:59:00,1,1429.83,17,17,2010-06-01,2009-12-03,174,0,1429.8300
4,12359.0,2010-03-23 08:55:00,2009-12-05 13:32:00,4,1522.23,66,65,2010-06-01,2009-12-03,69,107,380.5575


In [5]:
# Construct the 90-day repeat-purchase target

target_frames = []

for snapshot_date in snapshot_dates:

    target_end = snapshot_date + pd.Timedelta(days=90)

    # Identify customers purchasing during the target window
    target_customers = set(
        purchase_df.loc[
            (purchase_df["InvoiceDate"] >= snapshot_date) &
            (purchase_df["InvoiceDate"] < target_end),
            "Customer ID"
        ].unique()
    )

    # Select observations belonging to this snapshot
    snapshot_mask = (
        customer_features_df["snapshot_date"] == snapshot_date
    )

    snapshot_customers = customer_features_df.loc[
        snapshot_mask, "Customer ID"
    ]

    # Assign target labels
    labels = snapshot_customers.isin(target_customers).astype("int8")

    target_frames.append(
        pd.DataFrame({
            "Customer ID": snapshot_customers.values,
            "snapshot_date": snapshot_date,
            "repeat_purchase_90d": labels.values
        })
    )

# Combine target labels
target_df = pd.concat(target_frames, ignore_index=True)

# Merge targets into the feature dataset
modeling_df = customer_features_df.merge(
    target_df,
    on=["Customer ID", "snapshot_date"],
    how="left",
    validate="one_to_one"
)

# Validate target completeness
print("Modeling dataset shape:", modeling_df.shape)
print("Missing target values:", modeling_df["repeat_purchase_90d"].isna().sum())

print("\nTarget distribution:")
display(
    modeling_df["repeat_purchase_90d"]
    .value_counts()
    .rename(index={0: "No purchase", 1: "Repeat purchase"})
    .to_frame("observations")
)

print("\nTarget rate:")
print(f"{modeling_df['repeat_purchase_90d'].mean():.2%}")

# Compare target counts with the preliminary snapshot audit
target_validation = (
    modeling_df.groupby("snapshot_date")["repeat_purchase_90d"]
    .agg(
        eligible_customers="count",
        positive_customers="sum"
    )
    .reset_index()
)

target_validation["negative_customers"] = (
    target_validation["eligible_customers"] -
    target_validation["positive_customers"]
)

target_validation["positive_rate"] = (
    target_validation["positive_customers"] /
    target_validation["eligible_customers"]
)

comparison = target_validation.merge(
    snapshot_audit_df[
        [
            "snapshot_date",
            "eligible_customers",
            "positive_customers",
            "negative_customers"
        ]
    ],
    on="snapshot_date",
    suffixes=("_constructed", "_audit"),
    validate="one_to_one"
)

comparison["counts_match"] = (
    (comparison["eligible_customers_constructed"] ==
     comparison["eligible_customers_audit"]) &
    (comparison["positive_customers_constructed"] ==
     comparison["positive_customers_audit"]) &
    (comparison["negative_customers_constructed"] ==
     comparison["negative_customers_audit"])
)

print("\nSnapshot target validation:")
display(comparison)

print(
    "\nAll snapshot counts match:",
    comparison["counts_match"].all()
)

Modeling dataset shape: (47933, 13)
Missing target values: 0

Target distribution:


,observations
repeat_purchase_90d,
No purchase,24400
Repeat purchase,23533



Target rate:
49.10%

Snapshot target validation:


,snapshot_date,eligible_customers_constructed,positive_customers_constructed,negative_customers_constructed,positive_rate,eligible_customers_audit,positive_customers_audit,negative_customers_audit,counts_match
0,2010-06-01,2652,1327,1325,0.500377,2652,1327,1325,True
1,2010-07-01,2737,1394,1343,0.509317,2737,1394,1343,True
2,2010-08-01,2813,1604,1209,0.570210,2813,1604,1209,True
3,2010-09-01,2822,1750,1072,0.620128,2822,1750,1072,True
4,2010-10-01,2899,1680,1219,0.579510,2899,1680,1219,True
5,2010-11-01,3196,1577,1619,0.493429,3196,1577,1619,True
6,2010-12-01,3465,1318,2147,0.380375,3465,1318,2147,True
7,2011-01-01,3403,1284,2119,0.377314,3403,1284,2119,True
8,2011-02-01,3348,1328,2020,0.396655,3348,1328,2020,True
9,2011-03-01,3338,1414,1924,0.423607,3338,1414,1924,True



All snapshot counts match: True


In [8]:
feature_columns = [
    "recency_days",
    "frequency",
    "monetary",
    "purchase_line_count",
    "unique_products",
    "customer_history_days",
    "avg_invoice_value"
]

print(modeling_df[feature_columns].dtypes)

recency_days               int64
frequency                  int64
monetary                 float64
purchase_line_count        Int64
unique_products            int64
customer_history_days      int64
avg_invoice_value        float64
dtype: object


In [10]:
# Feature integrity and leakage checks

feature_columns = [
    "recency_days",
    "frequency",
    "monetary",
    "purchase_line_count",
    "unique_products",
    "customer_history_days",
    "avg_invoice_value"
]

# Temporary numeric representation for validation
numeric_features = modeling_df[feature_columns].to_numpy(
    dtype=np.float64,
    na_value=np.nan
)

feature_checks = {
    "All last purchases precede snapshot":
        (modeling_df["last_purchase_date"] < modeling_df["snapshot_date"]).all(),

    "All first purchases are within feature window":
        (modeling_df["first_purchase_date"] >= modeling_df["feature_start"]).all(),

    "All recency values are non-negative":
        (modeling_df["recency_days"] >= 0).all(),

    "Customer-snapshot pairs are unique":
        not modeling_df.duplicated(
            subset=["Customer ID", "snapshot_date"]
        ).any(),

    "No missing feature values":
        modeling_df[feature_columns].notna().all().all(),

    "No infinite feature values":
        np.isfinite(numeric_features).all(),

    "Target contains only binary values":
        modeling_df["repeat_purchase_90d"].isin([0, 1]).all()
}

for check, passed in feature_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print("\nTotal checks passed:", sum(feature_checks.values()))
print("Total checks:", len(feature_checks))

PASS: All last purchases precede snapshot
PASS: All first purchases are within feature window
PASS: All recency values are non-negative
PASS: Customer-snapshot pairs are unique
PASS: No missing feature values
PASS: No infinite feature values
PASS: Target contains only binary values

Total checks passed: 7
Total checks: 7


In [11]:
# Descriptive statistics of engineered features

feature_summary = modeling_df[feature_columns].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T

feature_summary["skewness"] = modeling_df[feature_columns].skew()
feature_summary["kurtosis"] = modeling_df[feature_columns].kurtosis()

display(feature_summary)

,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,max,skewness,kurtosis
recency_days,47933.0,64.692863,50.545448,0.0,0.0,4.0,21.0,52.0,102.0,144.0,161.0,176.0,179.0,0.600039,-0.804854
frequency,47933.0,2.853859,4.290121,1.0,1.0,1.0,1.0,2.0,3.0,6.0,8.0,18.0,117.0,8.991889,133.881427
monetary,47933.0,1291.276296,4639.983674,1.3,39.95,105.5,262.47,516.55,1125.39,2280.288,3725.268,12931.44,179248.38,18.179023,451.58447
purchase_line_count,47933.0,58.872989,108.596562,1.0,1.0,3.0,15.0,31.0,66.0,129.0,193.0,398.0,3372.0,10.985002,202.041756
unique_products,47933.0,45.435316,58.586172,1.0,1.0,3.0,14.0,29.0,56.0,101.0,140.0,248.0,1242.0,6.1994,76.370391
customer_history_days,47933.0,48.458557,57.900387,0.0,0.0,0.0,0.0,12.0,99.0,142.0,157.0,171.0,179.0,0.739442,-0.97785
avg_invoice_value,47933.0,396.221928,1006.795019,1.3,35.4,90.0,176.671667,293.445,431.49,690.48,960.41,2024.06,77183.6,57.389554,4255.597118


In [12]:
# Inspect extreme customer-snapshot observations

inspection_columns = [
    "Customer ID",
    "snapshot_date",
    "recency_days",
    "frequency",
    "monetary",
    "purchase_line_count",
    "unique_products",
    "avg_invoice_value"
]

print("Top 10 observations by monetary value:")

display(
    modeling_df.nlargest(10, "monetary")[inspection_columns]
)

print("\nTop 10 observations by average invoice value:")

display(
    modeling_df.nlargest(10, "avg_invoice_value")[inspection_columns]
)

Top 10 observations by monetary value:


,Customer ID,snapshot_date,recency_days,frequency,monetary,purchase_line_count,unique_products,avg_invoice_value
23901,18102.0,2011-01-01,22,40,179248.38,210,123,4481.209500
20495,18102.0,2010-12-01,15,40,166763.39,222,131,4169.084750
10934,18102.0,2010-09-01,0,56,165461.23,371,184,2954.664821
17035,18102.0,2010-11-01,14,41,157845.13,242,152,3849.881220
2572,18102.0,2010-06-01,10,44,152280.11,370,190,3460.911591
25282,14646.0,2011-02-01,10,41,149976.25,1136,479,3657.957317
13833,18102.0,2010-10-01,30,48,144246.12,307,161,3005.127500
28641,14646.0,2011-03-01,6,39,143537.79,1113,489,3680.456154
8113,18102.0,2010-08-01,1,46,142964.24,347,163,3107.918261
18463,14646.0,2010-12-01,0,40,137697.75,1015,474,3442.443750



Top 10 observations by average invoice value:


,Customer ID,snapshot_date,recency_days,frequency,monetary,purchase_line_count,unique_products,avg_invoice_value
23987,12346.0,2011-02-01,13,1,77183.6,1,1,77183.600000
27335,12346.0,2011-03-01,41,1,77183.6,1,1,77183.600000
30673,12346.0,2011-04-01,72,1,77183.6,1,1,77183.600000
33972,12346.0,2011-05-01,102,1,77183.6,1,1,77183.600000
37019,12346.0,2011-06-01,133,1,77183.6,1,1,77183.600000
39677,12346.0,2011-07-01,163,1,77183.6,1,1,77183.600000
43997,15749.0,2011-08-01,104,1,21535.9,5,5,21535.900000
46772,15749.0,2011-09-01,135,1,21535.9,5,5,21535.900000
35693,15749.0,2011-05-01,12,3,44534.3,10,5,14844.766667
38562,15749.0,2011-06-01,43,3,44534.3,10,5,14844.766667


In [13]:
# Inspect cancellation activity for extreme-value customers

investigation_customers = [12346, 15749, 18102, 14646]

cancellation_df = df.loc[
    df["transaction_category"].isin([
        "merchandise_cancellation_candidate"
    ])
].copy()

cancellation_summary = (
    cancellation_df[
        cancellation_df["Customer ID"].isin(investigation_customers)
    ]
    .groupby("Customer ID")
    .agg(
        cancellation_lines=("Invoice", "size"),
        cancellation_revenue=("merchandise_revenue", "sum"),
        first_cancellation=("InvoiceDate", "min"),
        last_cancellation=("InvoiceDate", "max")
    )
    .reset_index()
)

display(cancellation_summary)

,Customer ID,cancellation_lines,cancellation_revenue,first_cancellation,last_cancellation
0,12346.0,1,-77183.60,2011-01-18 10:17:00,2011-01-18 10:17:00
1,14646.0,29,-3548.78,2010-02-26 12:44:00,2011-08-31 11:31:00
2,15749.0,5,-22998.40,2011-04-18 13:08:00,2011-04-18 13:08:00
3,18102.0,4,-2578.40,2010-02-21 15:51:00,2010-07-27 16:54:00


In [14]:
# Construct historical cancellation features

cancellation_df = df.loc[
    df["transaction_category"].eq(
        "merchandise_cancellation_candidate"
    )
].copy()

cancellation_feature_frames = []

for snapshot_date in snapshot_dates:

    feature_start = snapshot_date - pd.Timedelta(days=180)

    window_cancellations = cancellation_df.loc[
        (cancellation_df["InvoiceDate"] >= feature_start) &
        (cancellation_df["InvoiceDate"] < snapshot_date)
    ].copy()

    cancellation_features = (
        window_cancellations.groupby("Customer ID")
        .agg(
            cancellation_line_count=("Invoice", "size"),
            cancellation_invoice_count=("Invoice", "nunique"),
            cancellation_value=("merchandise_revenue", lambda x: -x.sum())
        )
        .reset_index()
    )

    cancellation_features["snapshot_date"] = snapshot_date

    cancellation_feature_frames.append(cancellation_features)

cancellation_features_df = pd.concat(
    cancellation_feature_frames,
    ignore_index=True
)

# Merge cancellation features into the modeling dataset
modeling_df = modeling_df.merge(
    cancellation_features_df,
    on=["Customer ID", "snapshot_date"],
    how="left",
    validate="one_to_one"
)

# Customers without cancellation activity receive zero values
cancellation_columns = [
    "cancellation_line_count",
    "cancellation_invoice_count",
    "cancellation_value"
]

modeling_df[cancellation_columns] = (
    modeling_df[cancellation_columns].fillna(0)
)

print("Updated modeling dataset shape:", modeling_df.shape)

print("\nMissing cancellation feature values:")
print(modeling_df[cancellation_columns].isna().sum())

print("\nCancellation feature summary:")
display(modeling_df[cancellation_columns].describe().T)

Updated modeling dataset shape: (47933, 16)

Missing cancellation feature values:
cancellation_line_count       0
cancellation_invoice_count    0
cancellation_value            0
dtype: int64

Cancellation feature summary:


,count,mean,std,min,25%,50%,75%,max
cancellation_line_count,47933.0,1.335134,4.296932,0.0,0.0,0.0,1.0,113.0
cancellation_invoice_count,47933.0,0.580435,1.310358,0.0,0.0,0.0,1.0,29.0
cancellation_value,47933.0,46.505116,943.663743,0.0,0.0,0.0,10.45,77183.6


In [15]:
# Validate historical cancellation features

cancellation_checks = {
    "Cancellation features are non-negative":
        (modeling_df[cancellation_columns] >= 0).all().all(),

    "Cancellation line counts are integers":
        (modeling_df["cancellation_line_count"] % 1 == 0).all(),

    "Cancellation invoice counts are integers":
        (modeling_df["cancellation_invoice_count"] % 1 == 0).all(),

    "Cancellation invoice count does not exceed line count":
        (
            modeling_df["cancellation_invoice_count"] <=
            modeling_df["cancellation_line_count"]
        ).all(),

    "Customer-snapshot pairs remain unique":
        not modeling_df.duplicated(
            subset=["Customer ID", "snapshot_date"]
        ).any(),

    "No missing cancellation features":
        modeling_df[cancellation_columns].notna().all().all()
}

for check, passed in cancellation_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print("\nTotal checks passed:", sum(cancellation_checks.values()))
print("Total checks:", len(cancellation_checks))

PASS: Cancellation features are non-negative
PASS: Cancellation line counts are integers
PASS: Cancellation invoice counts are integers
PASS: Cancellation invoice count does not exceed line count
PASS: Customer-snapshot pairs remain unique
PASS: No missing cancellation features

Total checks passed: 6
Total checks: 6


In [16]:
# Create log-transformed feature versions

log_features = [
    "frequency",
    "monetary",
    "purchase_line_count",
    "unique_products",
    "avg_invoice_value",
    "cancellation_line_count",
    "cancellation_invoice_count",
    "cancellation_value"
]

for column in log_features:
    modeling_df[f"log1p_{column}"] = np.log1p(
        modeling_df[column]
    )

# Compare original and transformed skewness
transformation_summary = pd.DataFrame({
    "original_skewness": modeling_df[log_features].skew(),
    "log1p_skewness": modeling_df[
        [f"log1p_{column}" for column in log_features]
    ].skew().to_numpy()
}, index=log_features)

display(transformation_summary)

print("Updated modeling dataset shape:", modeling_df.shape)

,original_skewness,log1p_skewness
frequency,8.991889,1.551656
monetary,18.179023,0.346952
purchase_line_count,10.985002,-0.076375
unique_products,6.1994,-0.279876
avg_invoice_value,57.389554,-0.039822
cancellation_line_count,8.963863,1.870174
cancellation_invoice_count,5.917046,1.628872
cancellation_value,70.779325,1.387996


Updated modeling dataset shape: (47933, 24)


In [17]:
# Examine correlations among candidate features

candidate_features = [
    "recency_days",
    "customer_history_days",
    "log1p_frequency",
    "log1p_monetary",
    "log1p_purchase_line_count",
    "log1p_unique_products",
    "log1p_avg_invoice_value",
    "log1p_cancellation_line_count",
    "log1p_cancellation_invoice_count",
    "log1p_cancellation_value"
]

correlation_data = modeling_df[
    candidate_features + ["repeat_purchase_90d"]
]

correlation_matrix = correlation_data.corr(method="spearman")

display(
    correlation_matrix.round(3)
)

,recency_days,customer_history_days,log1p_frequency,log1p_monetary,log1p_purchase_line_count,log1p_unique_products,log1p_avg_invoice_value,log1p_cancellation_line_count,log1p_cancellation_invoice_count,log1p_cancellation_value,repeat_purchase_90d
recency_days,1.000,-0.570,-0.504,-0.417,-0.375,-0.335,-0.119,-0.203,-0.210,-0.198,-0.272
customer_history_days,-0.570,1.000,0.914,0.702,0.605,0.536,0.153,0.382,0.393,0.377,0.380
log1p_frequency,-0.504,0.914,1.000,0.744,0.649,0.573,0.126,0.420,0.433,0.418,0.390
log1p_monetary,-0.417,0.702,0.744,1.000,0.738,0.689,0.725,0.421,0.425,0.419,0.369
log1p_purchase_line_count,-0.375,0.605,0.649,0.738,1.000,0.987,0.433,0.337,0.337,0.319,0.320
log1p_unique_products,-0.335,0.536,0.573,0.689,0.987,1.000,0.433,0.309,0.307,0.289,0.289
log1p_avg_invoice_value,-0.119,0.153,0.126,0.725,0.433,0.433,1.000,0.210,0.203,0.212,0.160
log1p_cancellation_line_count,-0.203,0.382,0.420,0.421,0.337,0.309,0.210,1.000,0.988,0.983,0.212
log1p_cancellation_invoice_count,-0.210,0.393,0.433,0.425,0.337,0.307,0.203,0.988,1.000,0.979,0.218
log1p_cancellation_value,-0.198,0.377,0.418,0.419,0.319,0.289,0.212,0.983,0.979,1.000,0.206


In [18]:
# Define the initial baseline feature set

baseline_features = [
    "recency_days",
    "log1p_frequency",
    "log1p_monetary",
    "log1p_unique_products",
    "log1p_cancellation_value"
]

target_column = "repeat_purchase_90d"

modeling_columns = [
    "Customer ID",
    "snapshot_date",
    *baseline_features,
    target_column
]

baseline_df = modeling_df[modeling_columns].copy()

print("Baseline dataset shape:", baseline_df.shape)
print("\nBaseline features:")
for feature in baseline_features:
    print("-", feature)

print("\nMissing values:")
print(baseline_df.isna().sum())

print("\nTarget distribution:")
print(baseline_df[target_column].value_counts())

print("\nDuplicate customer-snapshot observations:",
      baseline_df.duplicated(
          subset=["Customer ID", "snapshot_date"]
      ).sum())

display(baseline_df.head())

Baseline dataset shape: (47933, 8)

Baseline features:
- recency_days
- log1p_frequency
- log1p_monetary
- log1p_unique_products
- log1p_cancellation_value

Missing values:
Customer ID                 0
snapshot_date               0
recency_days                0
log1p_frequency             0
log1p_monetary              0
log1p_unique_products       0
log1p_cancellation_value    0
repeat_purchase_90d         0
dtype: int64

Target distribution:
repeat_purchase_90d
0    24400
1    23533
Name: count, dtype: int64

Duplicate customer-snapshot observations: 0


,Customer ID,snapshot_date,recency_days,log1p_frequency,log1p_monetary,log1p_unique_products,log1p_cancellation_value,repeat_purchase_90d
0,12346.0,2010-06-01,90,1.098612,3.369018,1.945910,0.000000,1
1,12349.0,2010-06-01,32,0.693147,6.974965,3.850148,3.224858,0
2,12355.0,2010-06-01,10,0.693147,6.192792,3.135494,0.000000,0
3,12358.0,2010-06-01,174,0.693147,7.266010,2.890372,0.000000,1
4,12359.0,2010-06-01,69,1.609438,7.328588,4.189655,4.553877,1


In [19]:
# Create chronological train, validation, and test sets

train_dates = pd.date_range("2010-06-01", "2010-11-01", freq="MS")
validation_dates = pd.date_range("2011-02-01", "2011-04-01", freq="MS")
test_dates = pd.date_range("2011-08-01", "2011-09-01", freq="MS")

train_df = baseline_df[
    baseline_df["snapshot_date"].isin(train_dates)
].copy()

validation_df = baseline_df[
    baseline_df["snapshot_date"].isin(validation_dates)
].copy()

test_df = baseline_df[
    baseline_df["snapshot_date"].isin(test_dates)
].copy()

# Verify that all observations have been assigned as intended
assigned_observations = (
    len(train_df) + len(validation_df) + len(test_df)
)

print("Partition sizes:")
print("Training:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)
print("Total assigned observations:", assigned_observations)

# Summarize dates, target distribution, and customer coverage
def summarize_partition(name, partition):
    print(f"\n{name}")
    print("Snapshot dates:",
          sorted(partition["snapshot_date"].unique()))
    print("Observations:", len(partition))
    print("Distinct customers:",
          partition["Customer ID"].nunique())
    print("Positive rate:",
          f"{partition[target_column].mean():.2%}")
    print("Target counts:")
    print(partition[target_column].value_counts().sort_index())

summarize_partition("TRAINING", train_df)
summarize_partition("VALIDATION", validation_df)
summarize_partition("TEST", test_df)

# Check customer overlap across partitions
train_customers = set(train_df["Customer ID"])
validation_customers = set(validation_df["Customer ID"])
test_customers = set(test_df["Customer ID"])

print("\nCustomer overlap:")
print("Train and validation:",
      len(train_customers & validation_customers))
print("Train and test:",
      len(train_customers & test_customers))
print("Validation and test:",
      len(validation_customers & test_customers))

# Check that target windows do not overlap across partitions
print("\nTarget-window boundaries:")
print("Last training target end:",
      train_dates.max() + pd.Timedelta(days=90))
print("First validation snapshot:",
      validation_dates.min())
print("Last validation target end:",
      validation_dates.max() + pd.Timedelta(days=90))
print("First test snapshot:",
      test_dates.min())

Partition sizes:
Training: (17119, 8)
Validation: (9985, 8)
Test: (5534, 8)
Total assigned observations: 32638

TRAINING
Snapshot dates: [Timestamp('2010-06-01 00:00:00'), Timestamp('2010-07-01 00:00:00'), Timestamp('2010-08-01 00:00:00'), Timestamp('2010-09-01 00:00:00'), Timestamp('2010-10-01 00:00:00'), Timestamp('2010-11-01 00:00:00')]
Observations: 17119
Distinct customers: 3893
Positive rate: 54.51%
Target counts:
repeat_purchase_90d
0    7787
1    9332
Name: count, dtype: int64

VALIDATION
Snapshot dates: [Timestamp('2011-02-01 00:00:00'), Timestamp('2011-03-01 00:00:00'), Timestamp('2011-04-01 00:00:00')]
Observations: 9985
Distinct customers: 3745
Positive rate: 41.96%
Target counts:
repeat_purchase_90d
0    5795
1    4190
Name: count, dtype: int64

TEST
Snapshot dates: [Timestamp('2011-08-01 00:00:00'), Timestamp('2011-09-01 00:00:00')]
Observations: 5534
Distinct customers: 2968
Positive rate: 58.19%
Target counts:
repeat_purchase_90d
0    2314
1    3220
Name: count, dtype: 

In [20]:
# Record temporal partition definitions

partition_summary = pd.DataFrame([
    {
        "partition": "train",
        "first_snapshot": train_df["snapshot_date"].min(),
        "last_snapshot": train_df["snapshot_date"].max(),
        "observations": len(train_df),
        "distinct_customers": train_df["Customer ID"].nunique(),
        "positive_rate": train_df[target_column].mean()
    },
    {
        "partition": "validation",
        "first_snapshot": validation_df["snapshot_date"].min(),
        "last_snapshot": validation_df["snapshot_date"].max(),
        "observations": len(validation_df),
        "distinct_customers": validation_df["Customer ID"].nunique(),
        "positive_rate": validation_df[target_column].mean()
    },
    {
        "partition": "test",
        "first_snapshot": test_df["snapshot_date"].min(),
        "last_snapshot": test_df["snapshot_date"].max(),
        "observations": len(test_df),
        "distinct_customers": test_df["Customer ID"].nunique(),
        "positive_rate": test_df[target_column].mean()
    }
])

display(
    partition_summary.style.format({
        "positive_rate": "{:.2%}"
    })
)

print("Total observations:", len(baseline_df))
print("Assigned observations:", len(train_df) + len(validation_df) + len(test_df))
print("Excluded observations:",
      len(baseline_df) - len(train_df) - len(validation_df) - len(test_df))

,partition,first_snapshot,last_snapshot,observations,distinct_customers,positive_rate
0,train,2010-06-01 00:00:00,2010-11-01 00:00:00,17119,3893,54.51%
1,validation,2011-02-01 00:00:00,2011-04-01 00:00:00,9985,3745,41.96%
2,test,2011-08-01 00:00:00,2011-09-01 00:00:00,5534,2968,58.19%


Total observations: 47933
Assigned observations: 32638
Excluded observations: 15295


In [21]:
# Freeze the modeling dataset and temporal partitions

# Assign partition labels to every customer-snapshot observation
frozen_df = baseline_df.copy()

frozen_df["partition"] = "embargo"

frozen_df.loc[
    frozen_df["snapshot_date"].isin(train_dates), "partition"
] = "train"

frozen_df.loc[
    frozen_df["snapshot_date"].isin(validation_dates), "partition"
] = "validation"

frozen_df.loc[
    frozen_df["snapshot_date"].isin(test_dates), "partition"
] = "test"

# Define project-level output directory
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the complete baseline dataset
dataset_path = (
    OUTPUT_DIR / "customer_purchase_propensity_dataset.parquet"
)

frozen_df.to_parquet(dataset_path, index=False)

# Save partition summary
partition_path = (
    OUTPUT_DIR / "temporal_partition_summary.csv"
)

partition_summary.to_csv(partition_path, index=False)

# Validate saved dataset
saved_df = pd.read_parquet(dataset_path)

print("Saved dataset:", dataset_path)
print("Saved partition summary:", partition_path)

print("\nSaved dataset shape:", saved_df.shape)
print("\nPartition distribution:")
print(saved_df["partition"].value_counts())

print("\nDataset reconciliation:")
print("Original observations:", len(baseline_df))
print("Saved observations:", len(saved_df))
print("Observation counts match:", len(baseline_df) == len(saved_df))

print("\nPartition counts:")
print(
    saved_df.groupby("partition")["repeat_purchase_90d"]
    .agg(["count", "mean"])
)

Saved dataset: C:\Users\Kia\Downloads\DATA-SCIENCE-PORTFOLIO\customer-intelligence-revenue-optimization\data\processed\customer_purchase_propensity_dataset.parquet
Saved partition summary: C:\Users\Kia\Downloads\DATA-SCIENCE-PORTFOLIO\customer-intelligence-revenue-optimization\data\processed\temporal_partition_summary.csv

Saved dataset shape: (47933, 9)

Partition distribution:
partition
train         17119
embargo       15295
validation     9985
test           5534
Name: count, dtype: int64

Dataset reconciliation:
Original observations: 47933
Saved observations: 47933
Observation counts match: True

Partition counts:
            count      mean
partition                  
embargo     15295  0.444001
test         5534  0.581858
train       17119  0.545125
validation   9985  0.419629


In [22]:
# Final temporal leakage audit

# Check historical feature boundaries
historical_checks = {
    "All feature transactions precede their snapshot":
        (
            modeling_df["last_purchase_date"] <
            modeling_df["snapshot_date"]
        ).all(),

    "All feature windows begin at the defined 180-day boundary":
        (
            modeling_df["feature_start"] ==
            modeling_df["snapshot_date"] - pd.Timedelta(days=180)
        ).all(),

    "All feature observations have non-negative recency":
        (modeling_df["recency_days"] >= 0).all(),

    "No missing values in baseline features or target":
        frozen_df[
            baseline_features + [target_column]
        ].notna().all().all(),

    "Customer-snapshot pairs are unique":
        not frozen_df.duplicated(
            subset=["Customer ID", "snapshot_date"]
        ).any()
}

# Check target-window boundaries between partitions
train_target_end = train_dates.max() + pd.Timedelta(days=90)
validation_target_end = (
    validation_dates.max() + pd.Timedelta(days=90)
)

temporal_checks = {
    "Training target window ends before validation":
        train_target_end <= validation_dates.min(),

    "Validation target window ends before test":
        validation_target_end <= test_dates.min(),

    "Test target window is covered by available data":
        (
            test_dates.max() + pd.Timedelta(days=90)
            <= last_purchase_date.normalize()
        ),

    "All partition labels are valid":
        frozen_df["partition"].isin(
            ["train", "validation", "test", "embargo"]
        ).all(),

    "Saved dataset retains all observations":
        len(saved_df) == len(baseline_df),

    "Saved partition assignments match":
        saved_df["partition"].equals(frozen_df["partition"])
}

all_checks = {**historical_checks, **temporal_checks}

for check, passed in all_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print("\nTotal checks passed:", sum(all_checks.values()))
print("Total checks:", len(all_checks))

print("\nTarget-window boundaries:")
print("Training ends:", train_target_end)
print("Validation starts:", validation_dates.min())
print("Validation ends:", validation_target_end)
print("Test starts:", test_dates.min())
print(
    "Test target window ends:",
    test_dates.max() + pd.Timedelta(days=90)
)
print("Last available qualifying purchase:", last_purchase_date)

PASS: All feature transactions precede their snapshot
PASS: All feature windows begin at the defined 180-day boundary
PASS: All feature observations have non-negative recency
PASS: No missing values in baseline features or target
PASS: Customer-snapshot pairs are unique
PASS: Training target window ends before validation
PASS: Validation target window ends before test
PASS: Test target window is covered by available data
PASS: All partition labels are valid
PASS: Saved dataset retains all observations
PASS: Saved partition assignments match

Total checks passed: 11
Total checks: 11

Target-window boundaries:
Training ends: 2011-01-30 00:00:00
Validation starts: 2011-02-01 00:00:00
Validation ends: 2011-06-30 00:00:00
Test starts: 2011-08-01 00:00:00
Test target window ends: 2011-11-30 00:00:00
Last available qualifying purchase: 2011-12-09 12:50:00


In [23]:
# Save a complete summary of all temporal partitions

complete_partition_summary = (
    frozen_df.groupby("partition")
    .agg(
        first_snapshot=("snapshot_date", "min"),
        last_snapshot=("snapshot_date", "max"),
        observations=("Customer ID", "size"),
        distinct_customers=("Customer ID", "nunique"),
        positive_rate=(target_column, "mean")
    )
    .reindex(["train", "validation", "test", "embargo"])
    .reset_index()
)

display(
    complete_partition_summary.style.format({
        "positive_rate": "{:.2%}"
    })
)

# Overwrite the previous summary with the complete version
partition_path = (
    OUTPUT_DIR / "temporal_partition_summary.csv"
)

complete_partition_summary.to_csv(
    partition_path,
    index=False
)

# Read back and verify
saved_partition_summary = pd.read_csv(partition_path)

print("\nSaved:", partition_path)
print("Summary rows:", len(saved_partition_summary))
print(
    "Total observations represented:",
    saved_partition_summary["observations"].sum()
)

,partition,first_snapshot,last_snapshot,observations,distinct_customers,positive_rate
0,train,2010-06-01 00:00:00,2010-11-01 00:00:00,17119,3893,54.51%
1,validation,2011-02-01 00:00:00,2011-04-01 00:00:00,9985,3745,41.96%
2,test,2011-08-01 00:00:00,2011-09-01 00:00:00,5534,2968,58.19%
3,embargo,2010-12-01 00:00:00,2011-07-01 00:00:00,15295,4435,44.40%



Saved: C:\Users\Kia\Downloads\DATA-SCIENCE-PORTFOLIO\customer-intelligence-revenue-optimization\data\processed\temporal_partition_summary.csv
Summary rows: 4
Total observations represented: 47933
